### 0. Runtime sanity check
<small>
Эта ячейка нужна для самого первого вопроса: я вообще в правильной среде или нет?

Что она проверяет:
- текущую рабочую директорию,
- имя хоста,
- есть ли **CUDA**,
- виден ли `/content`,
- не сижу ли я случайно в локальной машине вместо **Colab** runtime,
- версию **Python**,
- версию **Torch**,
- **CUDA** runtime,
- имя **GPU**,
- число доступных **CPU**.

Это просто диагностическая ячейка перед любыми действиями.
Если здесь нет **GPU** или нет `/content`, дальше запускать пайплайн бессмысленно.
</small>

In [1]:
import os, sys, socket, torch, pathlib, shutil, subprocess, textwrap
print("cwd:", os.getcwd())
print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("content exists:", pathlib.Path("/content").exists())
print("local project exists:", pathlib.Path("/home/yaroslav").exists())
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA version :", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("CPU cores visible:", os.cpu_count())

cwd: /content
host: 8de84f8c85b0
cuda: True
content exists: True
local project exists: False
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Torch version: 2.10.0+cu128
CUDA version : 12.8
CUDA available: True
GPU name: Tesla T4
CPU cores visible: 2


### 1. Mount Google Drive
<small>
Проектовый репозиторий, старые <code>runs/</code>, архив **PDBBind** и <code>esm_cache/</code> лежат на **Google Drive**.

Эта ячейка просто подключает **Drive** к runtime, чтобы дальше можно было:
- подтянуть свежий код,
- скопировать нужные файлы в `/content`,
- потом вернуть артефакты обратно на **Drive**.

Важно: тренировка идет не на **Drive**, а в `/content`.
**Drive** здесь используется как долговременное хранилище, а не как рабочая файловая система для обучения.
</small>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Imports for notebook orchestration
<small>
Здесь только те импорты, которые нужны самому ноутбуку как "пульту управления":

- менять директории,
- запускать фоновые процессы,
- смотреть версию **Torch**.

Логика обучения, датасетов и моделей здесь не живет.
Она уже вынесена в проектные файлы и в `scripts/`.
</small>

In [3]:
import os
import subprocess
import torch

### 3. Define paths and refresh the Drive-side repo
<small>
Здесь задаются две важные директории:

- `SRC` — репозиторий на **Google Drive**,
- `DST` — временная рабочая копия в `/content`.

Почему так:
- код и обучение лучше держать в `/content`, потому что это быстрее и стабильнее,
- **Drive** остается местом хранения, а не местом активной работы.

Также здесь делается:
- переход в `SRC`,
- `git checkout -- config.json` чтобы сбросить временные правки в конфиге,
- `git pull` чтобы подтянуть свежий код.

Смысл простой: на **Drive** лежит "источник истины", а в `/content` каждый запуск собирается свежая рабочая копия.
</small>

In [4]:
SRC = "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs"
DST = "/content/DUMPLINGs"

LOCAL_RUNS = f"{DST}/runs"
DRIVE_RUNS = f"{SRC}/runs"

LOCAL_FEATURES = f"{DST}/protein_context_features"
DRIVE_FEATURES = f"{SRC}/protein_context_features"

os.chdir(SRC)
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 56.92 KiB | 798.00 KiB/s, done.
From https://github.com/BlackSabbitch/DUMPLINGs
   08abfd3..fbae4ea  main       -> origin/main
Updating 08abfd3..fbae4ea
Fast-forward
 config.json  |    4 +-
 main_1.ipynb | 3715 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++-
 2 files changed, 3692 insertions(+), 27 deletions(-)


### 4. Stage the workspace into `/content`
<small>
Это одна из самых важных стадий.

Скрипт:
- пересоздает рабочую папку `DST`,
- копирует туда только нужные проектные файлы,
- копирует `esm_cache/`, если он уже есть на **Drive**,
- копирует архив `pdbbind_v2016.tar.gz`.

Зачем это нужно:
- не работать напрямую из **Google Drive**,
- не тянуть за собой весь возможный мусор,
- иметь чистую воспроизводимую рабочую копию на каждый run.

То есть после этой ячейки `/content/DUMPLINGs` — это реальное место, из которого будет запускаться эксперимент.
</small>

In [5]:
!python scripts/colab_stage_workspace.py \
  --src "{SRC}" \
  --dst "{DST}" \
  --drive-runs "{DRIVE_RUNS}"

Copied protein context features from /content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/protein_context_features (2579 top-level entries, source=protein_context_features)
Copied archive to /content/DUMPLINGs/pdbbind_v2016.tar.gz
Workspace staged into /content/DUMPLINGs


### 5. Enter the staged workspace
<small>
После копирования мы явно переходим в <code>DST</code>.

Это важно, потому что дальше:
- `run.sh`,
- `run.py`,
- относительные пути вроде `datasets/`, `runs/`, `esm_cache/`

должны разрешаться именно внутри `/content/DUMPLINGs`, а не где-то еще.
</small>

In [6]:
os.chdir(DST)
print(os.getcwd())

/content/DUMPLINGs


### 6. Install ordinary project requirements
<small>
Эта ячейка ставит обычные зависимости проекта из <code>requirements.txt</code>.

Сюда относятся, например:
- `pandas`
- `tqdm`
- `matplotlib`
- `biopython`
- `rdkit`
- `fair-esm`
- `uniplot`

Почему это вынесено в отдельный шаг:
- это обычные project dependencies,
- они логически отличаются от **PyG/Torch-Geometric** стека,
- их удобнее диагностировать отдельно.

Важно: здесь не ставится специальный **CUDA/PyG** стек.
Он ставится следующим шагом отдельным скриптом, потому что там нужна привязка к конкретным версиям **Torch** и **CUDA** в **Colab** runtime.
</small>

In [7]:
!python scripts/colab_install_requirements.py

Requirements file: requirements.txt
Skipping prefixes:
  - torch
  - torch-geometric
  - torch-scatter
  - torch-sparse
  - torch-cluster
  - pyg-lib
Installing filtered requirements:
  - rdkit
  - biopython
  - pandas
  - tqdm
  - matplotlib
  - uniplot
  - fair-esm


### 7. Install Colab-specific PyG dependencies
<small>
Эта стадия не про сам проект, а про среду выполнения.

Скрипт:
- определяет текущую версию **Torch** и **CUDA** в runtime,
- подбирает правильный wheel index для **PyG**,
- сносит потенциально несовместимые старые пакеты,
- ставит совместимые `pyg-lib`, `torch-scatter`, `torch-sparse`, `torch-cluster`, `torch-geometric`,
- делает небольшую проверку, что **CUDA**-часть действительно работает.

Это именно **Colab** bootstrap.
В обычной локальной среде такой шаг может не понадобиться.
</small>

In [8]:
!python scripts/colab_install_pyg.py

PyG wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
$ pip uninstall -y pyg-lib torch-scatter torch-sparse torch-cluster torch-geometric || true
Found existing installation: pyg_lib 0.6.0+pt210cu128
Uninstalling pyg_lib-0.6.0+pt210cu128:
  Successfully uninstalled pyg_lib-0.6.0+pt210cu128
Found existing installation: torch_scatter 2.1.2+pt210cu128
Uninstalling torch_scatter-2.1.2+pt210cu128:
  Successfully uninstalled torch_scatter-2.1.2+pt210cu128
Found existing installation: torch_sparse 0.6.18+pt210cu128
Uninstalling torch_sparse-0.6.18+pt210cu128:
  Successfully uninstalled torch_sparse-0.6.18+pt210cu128
Found existing installation: torch_cluster 1.6.3+pt210cu128
Uninstalling torch_cluster-1.6.3+pt210cu128:
  Successfully uninstalled torch_cluster-1.6.3+pt210cu128
Found existing installation: torch-geometric 2.7.0
Uninstalling torch-geometric-2.7.0:
  Successfully uninstalled torch-geometric-2.7.0
$ pip install pyg-lib torch-scatter torch-sparse torch-cluster -f https:

### 8. Re-check Torch and CUDA after dependency setup
<small>
После установки **PyG** полезно еще раз увидеть:

- что **Torch** жив,
- что **CUDA** по-прежнему доступна,
- что версия **CUDA** выглядит ожидаемо.

Это не обязательная ячейка, но удобная контрольная точка, если потом что-то идет странно.
</small>

In [9]:
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)

torch: 2.10.0+cu128
cuda available: True
cuda version: 12.8


### 9. Start background sync for `runs/`
<small>
Во время обучения <code>runs/</code> стоит периодически синкать обратно на **Drive**.

Зачем:
- если runtime упадет,
- если браузер/сессия отвалится,
- если **Colab** решит закончить жизнь раньше, чем хотелось,

то хотя бы часть логов и артефактов уже будет на **Drive**.

Почему здесь синкается только `runs/`, а не `esm_cache/`:
- `runs/` постоянно пополняется логами, history, checkpoint-ами,
- `esm_cache/` в обычной жизни достаточно:
  - скопировать в начале,
  - и один раз вернуть в конце.

То есть `runs/` — это живой поток артефактов,
а `esm_cache/` — накопительный кэш, который обычно не требует постоянной синхронизации.
</small>

In [10]:
sync_proc = subprocess.Popen([
    "bash",
    "scripts/colab_start_sync.sh",
    LOCAL_RUNS,
    DRIVE_RUNS,
])
print("Sync PID:", sync_proc.pid)

Sync PID: 18257


### 10. Launch the experiment
<small>
Это уже собственно запуск проекта.

Мы:
- делаем `run.sh` исполняемым,
- запускаем его с `--extract`.

Что означает `--extract`:
- перед построением датасета будет выполнена распаковка/подготовка исходного архива.

Если extraction не нужен, можно запускать просто `./run.sh`.

Важно: к этому моменту весь подготовительный orchestration уже должен быть завершен.
После этой ячейки начинает работать сам проектный pipeline.
</small>

In [11]:
!chmod +x run.sh
!./run.sh --extract

[INFO][EXPERIMENT] Starting experiment: DUMPLING_A15_logging_test_debug_mode_False
[INFO][EXPERIMENT] ================================================== *** STAGE: INITIALIZING *** ==================================================
[INFO][EXPERIMENT] Experiment signature: DUMPLING_A15_logging_test_debug_mode_False_20260507_165436
[INFO][EXPERIMENT] Base Datasets folder: datasets
[INFO][EXPERIMENT] Base protein context features folder: protein_context_features
[INFO][EXPERIMENT] Run results folder: runs/DUMPLING_A15_logging_test_debug_mode_False_20260507_165436
[INFO][EXPERIMENT] Log file: runs/DUMPLING_A15_logging_test_debug_mode_False_20260507_165436/log.txt
[INFO][EXPERIMENT] ================================================== *** STAGE: DATASET *** ==================================================
[INFO][InteractionGraphParser] Initialized dist_threshold=5.0, ca_only=False
[INFO][CNNParser] Initialized is_ligand=False
[INFO][REGISTRY] Loaded bad complexes registry from bad_complexes

### 11. Inspect local outputs after the run
<small>
Эта ячейка нужна, чтобы быстро посмотреть:

- жив ли `esm_cache/`,
- сколько там файлов,
- какие папки появились в `runs/`.

Это просто быстрая sanity-check стадия перед финальным sync обратно на **Drive**.
</small>

In [12]:
!du -sh /content/DUMPLINGs/protein_context_features 2>/dev/null || echo "protein_context_features is absent"
!find /content/DUMPLINGs/protein_context_features -type f | wc -l
!ls -lt /content/DUMPLINGs/runs | head

21M	/content/DUMPLINGs/protein_context_features
2579
total 4
drwxr-xr-x 2 root root 4096 May  7 17:05 DUMPLING_A15_logging_test_debug_mode_False_20260507_165436


### 12. Final sync back to Drive
<small>
В конце мы явно сохраняем:

- `runs/`
- `esm_cache/`

Почему `esm_cache/` достаточно синкать здесь один раз:
- эмбеддинги после записи не переписываются,
- в процессе рана просто могут появиться новые cache-файлы,
- поэтому финальный sync в нормальном сценарии достаточен.

Это последний шаг сохранения результатов из временного `/content` обратно в долговременное хранилище на **Drive**.
</small>

In [13]:
!bash scripts/colab_finalize_sync.sh \
  "/content/DUMPLINGs/runs" \
  "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/runs" \
  "/content/DUMPLINGs/protein_context_features" \
  "/content/drive/MyDrive/lisboa_ciencia_de_dados/2_sem/DUMPLINGs/protein_context_features"

### 13. Stop background sync
<small>
После завершения run можно явно остановить фоновый процесс синхронизации <code>runs/</code>.

Это не всегда критично, но полезно, чтобы не оставлять лишний висящий loop в runtime.
</small>

In [14]:
sync_proc.terminate()
print("Stopped sync PID:", sync_proc.pid)

Stopped sync PID: 18257
